In [ ]:
import os
import numpy as np
import torch
from datasets import ClassLabel, Dataset
from torch import nn
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)
# Try one of these in place of MODEL_NAME:
#MODEL_NAME = "microsoft/deberta-v3-small"
#MODEL_NAME = "BAAI/bge-base-en-v1.5"
#MODEL_NAME = "roberta-base"
MODEL_NAME="cross-encoder/ms-marco-MiniLM-L-6-v2"
OUTPUT_DIR = "./ms-marco-MiniLM-L-6-v2-qa-scorer"



SEED = 42

PROMPT_TYPES = [
    "Zero shot",
    "One shot",
    "Few shot",
    "Adaptive",
    "Prompt chaining",
    "Self consistency prompting",
    "Contrastive prompting",
    "Meta prompting",
    "Reverse prompting",
]


def build_raw_samples() -> list[dict]:
    """
    54 records (6 per prompting technique across high, medium, and low score tiers)
    evaluating conceptual "What is ... prompting?" answers on a 0.0 - 10.0 scale.
    """
    return [
        # 1. Zero shot (6 samples)
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "response": "Zero shot prompting asks a model to perform a task directly from instructions alone without providing any labeled examples or demonstrations in the prompt.",
            "score": 9.5,
        },
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "response": "In zero shot prompting, the model relies on its pre-trained knowledge and task instructions with zero in-context examples included.",
            "score": 9.5,
        },
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "response": "Zero shot prompting gives the model a few examples and asks for a rough guess.",
            "score": 1.0,
        },
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "response": "Zero shot prompting is when you don't write a prompt and let the model read your mind.",
            "score": 0,
        },
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "response": "It means giving the model ten solved examples before asking a question.",
            "score": 1.0,
        },
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "response": "Zero shot prompting sets temperature to zero so the output never changes.",
            "score": 0.0,
        },
        # 2. One shot (6 samples)
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "response": "One shot prompting provides a single example input-output demonstration alongside the task instructions so the model sees the expected format or pattern before answering.",
            "score": 9.8,
        },
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "response": "One shot prompting includes one sample pair in the prompt to illustrate the desired output structure before the model answers the real query.",
            "score": 9.8,
        },
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "response": "One shot prompting uses examples in a prompt to help the model understand the task.",
            "score": 7.2,
        },
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "response": "One shot prompting adds a single example, though it doesn't mention how it guides output format.",
            "score": 8.6,
        },
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "response": "One shot prompting runs the prompt once every ten minutes on a cron job.",
            "score": 0.0,
        },
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "response": "One shot prompting means you can only send one message per day to the API.",
            "score": 0.0,
        },
        # 3. Few shot (6 samples)
        {
            "prompt_type": "Few shot",
            "question": "What is few shot prompting?",
            "response": "Few shot prompting supplies multiple diverse examples in the context window to guide formatting, edge cases, and reasoning style for a target query without weight updates.",
            "score": 9.1,
        },
        {
            "prompt_type": "Few shot",
            "question": "What is few shot prompting?",
            "response": "Few shot prompting provides several input-output examples inside the prompt so the model generalizes the pattern in-context without fine-tuning.",
            "score": 8.9,
        },
        {
            "prompt_type": "Few shot",
            "question": "What is few shot prompting?",
            "response": "Few shot prompting adds examples to the prompt, though it is similar to one shot prompting.",
            "score": 8.5,
        },
        {
            "prompt_type": "Few shot",
            "question": "What is few shot prompting?",
            "response": "Few shot prompting gives multiple examples to show the pattern, without explaining edge cases or no-fine-tuning.",
            "score": 7.0,
        },
        {
            "prompt_type": "Few shot",
            "question": "What is few shot prompting?",
            "response": "Few shot prompting retrains all model weights on a small dataset.",
            "score": 0.0,
        },
        {
            "prompt_type": "Few shot",
            "question": "What is few shot prompting?",
            "response": "Few shot prompting means firing a model after a few bad responses.",
            "score": 0.0,
        },
        # 4. Adaptive (6 samples)
        {
            "prompt_type": "Adaptive",
            "question": "What is adaptive prompting?",
            "response": "Adaptive prompting dynamically adjusts prompt phrasing, detail level, or strategy based on user expertise, task difficulty, or intermediate model feedback.",
            "score": 9.0,
        },
        {
            "prompt_type": "Adaptive",
            "question": "What is adaptive prompting?",
            "response": "Adaptive prompting tailors instructions, complexity, or reasoning steps on the fly in response to context, difficulty, or prior outputs.",
            "score": 8.7,
        },
        {
            "prompt_type": "Adaptive",
            "question": "What is adaptive prompting?",
            "response": "Adaptive prompting changes how a prompt is written depending on the situation.",
            "score": 0.5,
        },
        {
            "prompt_type": "Adaptive",
            "question": "What is adaptive prompting?",
            "response": "Adaptive prompting adapts the tone of a reply, without covering difficulty or feedback loops.",
            "score": 0.2,
        },
        {
            "prompt_type": "Adaptive",
            "question": "What is adaptive prompting?",
            "response": "Adaptive prompting uses the exact same rigid template for every user and every question.",
            "score": 0.0,
        },
        {
            "prompt_type": "Adaptive",
            "question": "What is adaptive prompting?",
            "response": "Adaptive prompting adjusts the monitor stand height for ergonomics.",
            "score": 0.0,
        },
        # 5. Prompt chaining (6 samples)
        {
            "prompt_type": "Prompt chaining",
            "question": "What is prompt chaining?",
            "response": "Prompt chaining breaks a complex task into sequential sub-prompts where the output of one step becomes the structured input to the next step.",
            "score": 9.4,
        },
        {
            "prompt_type": "Prompt chaining",
            "question": "What is prompt chaining?",
            "response": "Prompt chaining decomposes multi-stage workflows into smaller prompts passed sequentially so each step stays focused and verifiable.",
            "score": 8.9,
        },
        {
            "prompt_type": "Prompt chaining",
            "question": "What is prompt chaining?",
            "response": "Prompt chaining splits a big task into multiple prompts instead of using one giant prompt.",
            "score": 8.5,
        },
        {
            "prompt_type": "Prompt chaining",
            "question": "What is prompt chaining?",
            "response": "Prompt chaining uses multiple prompts in order, though it doesn't mention passing outputs forward.",
            "score": 3.1,
        },
        {
            "prompt_type": "Prompt chaining",
            "question": "What is prompt chaining?",
            "response": "Prompt chaining links several prompts together with physical metal chains.",
            "score": 0.5,
        },
        {
            "prompt_type": "Prompt chaining",
            "question": "What is prompt chaining?",
            "response": "Prompt chaining locks the model into repeating the same sentence forever.",
            "score": 0.0,
        },
        # 6. Self consistency prompting (6 samples)
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "response": "Self consistency prompting samples multiple diverse reasoning paths for the same prompt and selects the most consistent final answer—often via majority vote—rather than relying on a single greedy path.",
            "score": 9.5,
        },
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "response": "Self consistency prompting generates several reasoning traces and aggregates their final answers by majority consensus to improve reliability on multi-step tasks.",
            "score": 9.0,
        },
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "response": "Self consistency prompting tries more than one reasoning attempt to see which answer comes up most.",
            "score": 9.8,
        },
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "response": "Self consistency prompting runs a prompt multiple times to check for agreement.",
            "score": 2.0,
        },
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "response": "Self consistency prompting asks the model to agree with whatever false claim the user makes.",
            "score": 0.0,
        },
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "response": "Self consistency prompting forces the model to give the exact same typo every time.",
            "score": 0.0,
        },
        # 7. Contrastive prompting (6 samples)
        {
            "prompt_type": "Contrastive prompting",
            "question": "What is contrastive prompting?",
            "response": "Contrastive prompting shows both desirable and undesirable examples—or contrasts valid vs. invalid reasoning—so the model learns clear boundaries for what to emulate and what to avoid.",
            "score": 9.2,
        },
        {
            "prompt_type": "Contrastive prompting",
            "question": "What is contrastive prompting?",
            "response": "Contrastive prompting pairs positive and negative demonstrations—or contrasting reasoning paths—to clarify what behavior to follow and what pitfalls to reject.",
            "score": 9.8,
        },
        {
            "prompt_type": "Contrastive prompting",
            "question": "What is contrastive prompting?",
            "response": "Contrastive prompting compares good and bad examples in the prompt.",
            "score": 3.5,
        },
        {
            "prompt_type": "Contrastive prompting",
            "question": "What is contrastive prompting?",
            "response": "Contrastive prompting includes negative examples to warn the model what not to do.",
            "score": 8.3,
        },
        {
            "prompt_type": "Contrastive prompting",
            "question": "What is contrastive prompting?",
            "response": "Contrastive prompting only changes screen brightness and contrast settings.",
            "score": 0.0,
        },
        {
            "prompt_type": "Contrastive prompting",
            "question": "What is contrastive prompting?",
            "response": "Contrastive prompting means arguing against the user in every reply.",
            "score": 0.0,
        },
        # 8. Meta prompting (6 samples)
        {
            "prompt_type": "Meta prompting",
            "question": "What is meta prompting?",
            "response": "Meta prompting uses a model to design, critique, refine, or generate prompts, rubrics, or problem-solving templates for itself or another workflow.",
            "score": 9.1,
        },
        {
            "prompt_type": "Meta prompting",
            "question": "What is meta prompting?",
            "response": "Meta prompting treats prompting itself as the task—having an LLM write, optimize, or structure prompts and evaluation criteria for downstream queries.",
            "score": 8.9,
        },
        {
            "prompt_type": "Meta prompting",
            "question": "What is meta prompting?",
            "response": "Meta prompting involves using AI to help write or improve prompts.",
            "score": 2.4,
        },
        {
            "prompt_type": "Meta prompting",
            "question": "What is meta prompting?",
            "response": "Meta prompting asks the model to review a prompt's wording without mentioning templates or workflows.",
            "score": 0.0,
        },
        {
            "prompt_type": "Meta prompting",
            "question": "What is meta prompting?",
            "response": "Meta prompting is only used when talking about Facebook company news.",
            "score": 0.0,
        },
        {
            "prompt_type": "Meta prompting",
            "question": "What is meta prompting?",
            "response": "Meta prompting puts metadata tags inside HTML headers for search engines.",
            "score": 0.2,
        },
        # 9. Reverse prompting (6 samples)
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "response": "Reverse prompting flips the interaction so the model asks targeted clarifying questions to uncover missing requirements before generating a final solution.",
            "score": 9.3,
        },
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "response": "Reverse prompting has the assistant interview the user with focused questions to gather constraints and intent before answering.",
            "score": 8.8,
        },
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "response": "Reverse prompting has the model ask questions back to the user before answering.",
            "score": 0.0,
        },
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "response": "Reverse prompting means asking the model to double-check its answer after responding.",
            "score": 0.0,
        },
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "response": "Reverse prompting spells every word in the prompt backward from right to left.",
            "score": 0.0,
        },
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "response": "Reverse prompting sends the response first and the question afterward.",
            "score": 0.0,
        },
    ]


def score_to_centered(score_0_10: float) -> float:
    """Map [0.0, 10.0] -> [-1.0, 1.0] so 5.0 sits at 0.0."""
    clamped = float(np.clip(score_0_10, 0.0, 10.0))
    return (clamped - 5.0) / 5.0


def centered_to_score(pred_centered: float) -> float:
    """Map [-1.0, 1.0] -> [0.0, 10.0]."""
    score = (float(pred_centered) * 5.0) + 5.0
    return float(np.clip(score, 0.0, 10.0))


def build_formatted_dataset() -> Dataset:
    rows = []
    for idx, item in enumerate(build_raw_samples()):
        text = (
            f"Prompting Topic: {item['prompt_type']}\n"
            f"Question: {item['question']}\n"
            f"Response: {item['response']}"
        )
        rows.append(
            {
                "id": idx,
                "prompt_type": item["prompt_type"],
                "text": text,
                "label": score_to_centered(item["score"]),
            }
        )
    return Dataset.from_list(rows)


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds_10 = np.array([centered_to_score(p) for p in preds.squeeze(-1)])
    labels_10 = np.array([centered_to_score(y) for y in labels.squeeze()])
    mse = float(np.mean((labels_10 - preds_10) ** 2))
    mae = float(np.mean(np.abs(labels_10 - preds_10)))
    pred_std = float(np.std(preds_10))
    return {
        "mse": mse,
        "rmse": float(np.sqrt(mse)),
        "mae": mae,
        "pred_std": pred_std,
    }


def get_precision_flags() -> dict:
    """Configure precision cleanly across CPU and CUDA GPUs."""
    if not torch.cuda.is_available():
        return {"use_cpu": True, "bf16": False, "fp16": False}
    bf16_supported = torch.cuda.is_bf16_supported()
    return {
        "use_cpu": False,
        "bf16": bf16_supported,
        "fp16": False,
    }


class CenteredRegressionTrainer(Trainer):
    """Uses SmoothL1Loss to stabilize regression on small datasets."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").float().view(-1)
        outputs = model(**inputs)
        logits = outputs.logits.float().view(-1)
        loss = nn.SmoothL1Loss(beta=0.2)(logits, labels)
        return (loss, outputs) if return_outputs else loss


def train_model():
    set_seed(SEED)
    precision = get_precision_flags()
    device_label = "GPU (CUDA)" if not precision["use_cpu"] else "CPU"
    print(f"Training on: {device_label} | precision={precision}")

    dataset = build_formatted_dataset().cast_column(
        "prompt_type", ClassLabel(names=PROMPT_TYPES)
    )
    split = dataset.train_test_split(
        test_size=0.2,
        seed=SEED,
        stratify_by_column="prompt_type",
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize_fn(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=512,
            padding="max_length",
        )

    tokenized_train = split["train"].map(tokenize_fn, batched=True)
    tokenized_eval = split["test"].map(tokenize_fn, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=1,
        problem_type="regression",
    )

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=3e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=24,
        weight_decay=0.01,
        warmup_steps=0.05,
        max_grad_norm=1.0,
        load_best_model_at_end=True,
        metric_for_best_model="mae",
        greater_is_better=False,
        logging_steps=5,
        report_to="none",
        seed=SEED,
        use_cpu=precision["use_cpu"],
        bf16=precision["bf16"],
        fp16=precision["fp16"],
    )

    trainer = CenteredRegressionTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    eval_results = trainer.evaluate()
    print("\nValidation metrics (0-10 scale):", eval_results)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"Saved model and tokenizer to {OUTPUT_DIR}")


def score_responses(model_dir: str, test_cases: list[dict]):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    model.eval()

    print("\n--- Post-Training Sample Evaluations (Scale: 0 - 10) ---")
    for i, case in enumerate(test_cases, start=1):
        text = (
            f"Prompting Topic: {case['prompt_type']}\n"
            f"Question: {case['question']}\n"
            f"Response: {case['response']}"
        )
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512,
        ).to(device)

        with torch.no_grad():
            pred_centered = model(**inputs).logits.squeeze().item()

        final_score = centered_to_score(pred_centered)
        print(f"\nSample {i} [{case['prompt_type']}]")
        print(f"  Question : {case['question']}")
        print(f"  Response : {case['response']}")
        print(f"  Score    : {final_score:.2f} / 10.00")


if __name__ == "__main__":
    train_model()

    sample_tests = [
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "response": "Zero shot prompting asks a model to complete a task directly from instructions without any examples included in the prompt.",
        },
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "response": "One shot prompting gives the model one example of how to map an input to an output before asking it to handle a new input.",
        },
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "response": "It deletes the system prompt and randomly drops words from the user question.",
        },
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "response": "Reverse prompting has the assistant ask clarifying questions to gather missing context before delivering the final answer.",
        },
    ]

    score_responses(OUTPUT_DIR, sample_tests)





In [ ]:
import os
import numpy as np
import torch
from datasets import Dataset
from torch import nn
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)

MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
OUTPUT_DIR = "./minilm-prompting-reference-scorer"
SEED = 42

def build_raw_samples() -> list[dict]:
    """
    9 detailed 10/10 reference answers (~150-200 words each), one per prompting technique.
    """
    return [
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "reference_10_of_10": (
                "Zero shot prompting is a technique in which a language model is asked to perform a task directly "
                "from natural language instructions without any labeled examples or demonstrations provided in the prompt. "
                "Instead of relying on in-context examples to show desired input-output patterns, the model draws on "
                "its pre-trained knowledge and instruction-following ability to interpret the task specification. "
                "A thorough answer explains both what is absent—zero examples—and what drives execution—clear task "
                "instructions, constraints, and existing model capabilities. It is especially useful when labeled "
                "examples are unavailable, expensive to create, or unnecessary for straightforward tasks such as "
                "basic classification, summarization, translation, or general question answering. However, zero shot "
                "prompting can struggle with complex formatting rules, niche domain conventions, or multi-step "
                "reasoning where demonstrations clarify expectations. Therefore, effective zero shot prompts state the "
                "objective unambiguously, specify output structure or tone when needed, and define boundaries so the "
                "model does not have to infer unstated requirements."
            ),
            "score": 10.0,
        },
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "reference_10_of_10": (
                "One shot prompting provides a single complete example—typically an input paired with its ideal "
                "output—alongside task instructions before asking the model to process a new input. That lone "
                "demonstration acts as a concrete blueprint showing desired formatting, style, level of detail, or "
                "transformation logic without requiring parameter updates or fine-tuning. A strong explanation highlights "
                "the distinction between zero examples and a single demonstration, as well as how one example reduces "
                "ambiguity when instructions alone might be interpreted in multiple ways. For instance, one example can "
                "illustrate a specific JSON schema, tone, or label convention immediately. At the same time, a complete "
                "answer notes its limitations: a single example may not cover diverse edge cases or complex variations, "
                "and a poorly chosen or unrepresentative example can bias the model toward a narrow pattern. Thus, "
                "one shot prompting works best when one clear, representative example captures the core task pattern."
            ),
            "score": 10.0,
        },
        {
            "prompt_type": "Few shot",
            "question": "What is few shot prompting?",
            "reference_10_of_10": (
                "Few shot prompting supplies multiple examples—usually several input-output pairs—inside the prompt "
                "context so the model can infer patterns, formatting rules, and reasoning style in-context without "
                "modifying model weights. A comprehensive answer emphasizes why multiple examples matter more than one: "
                "they help illustrate variety, edge cases, exceptions, and subtle distinctions across classes or "
                "formats. Few shot prompting leverages in-context learning while keeping the underlying model frozen, "
                "making it faster and more flexible than task-specific fine-tuning when requirements evolve. "
                "Good practice includes selecting diverse, accurate, and well-ordered examples that match the target "
                "task rather than noisy or contradictory ones. A complete explanation also recognizes trade-offs, "
                "such as increased token usage, higher latency and cost on long prompts, and potential sensitivity to "
                "example ordering or context-window limits. Overall, few shot prompting bridges direct instructions and "
                "fine-tuning when moderate guidance is needed."
            ),
            "score": 10.0,
        },
        {
            "prompt_type": "Adaptive",
            "question": "What is adaptive prompting?",
            "reference_10_of_10": (
                "Adaptive prompting dynamically adjusts prompt instructions, framing, detail level, or reasoning strategy "
                "based on changing context such as user expertise, task complexity, input characteristics, or feedback "
                "from earlier steps. Rather than applying a single static prompt to every request, an adaptive approach "
                "tailors how the model is guided so explanations or solutions fit the situation. A thorough answer "
                "explains what adapts—such as vocabulary depth, step-by-step scaffolding, format, or verification "
                "checks—and what triggers the adaptation, such as a beginner versus expert user, an ambiguous query, "
                "or an error detected during execution. It can be implemented through routing rules, dynamic prompt "
                "templates, or iterative refinement loops. Key benefits include better usability, clearer explanations, "
                "and more efficient use of tokens and reasoning effort across varied queries. A complete response also "
                "notes that added routing or adaptation logic introduces orchestration complexity that should be tested "
                "so behavior remains predictable."
            ),
            "score": 10.0,
        },
        {
            "prompt_type": "Prompt chaining",
            "question": "What is prompt chaining?",
            "reference_10_of_10": (
                "Prompt chaining decomposes a complex task into a sequence of smaller, focused prompts where the output "
                "of one step becomes the input—often structured—to the next step in a pipeline. Instead of forcing a "
                "single prompt to handle extraction, analysis, formatting, and validation all at once, chaining "
                "separates distinct sub-tasks into manageable stages. A strong explanation highlights sequential data "
                "flow between steps and why modularizing prompts improves reliability. Breaking work into stages makes "
                "it easier to inspect intermediate outputs, catch errors early, apply targeted instructions or "
                "constraints to each sub-task, and keep individual prompts concise and clear. It is widely used in "
                "multi-step workflows such as document summarization, research synthesis, data transformation, and code "
                "generation pipelines. A complete answer also notes trade-offs: chaining adds latency across "
                "sequential calls and can propagate errors if an earlier stage produces flawed output that is not "
                "checked before moving forward."
            ),
            "score": 10.0,
        },
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "reference_10_of_10": (
                "Self consistency prompting improves reliability on reasoning tasks by sampling multiple diverse "
                "reasoning paths for the same prompt and selecting the most consistent final answer, typically via "
                "majority vote. Rather than relying on a single greedy decoding path that might take a flawed "
                "intermediate step, this technique explores alternative ways to reach a solution and checks where "
                "independent paths converge. A thorough answer explains both parts of the mechanism: generating "
                "multiple reasoning traces (usually with non-zero sampling temperature) and aggregating their final "
                "answers by consensus. It is particularly effective for arithmetic, logic, and multi-step problem "
                "solving where multiple valid reasoning routes can lead to the same answer. A complete explanation "
                "also notes practical costs, since running multiple samples multiplies token usage and inference "
                "latency, and majority voting assumes the correct answer is more common across sampled paths than "
                "any single incorrect one."
            ),
            "score": 10.0,
        },
        {
            "prompt_type": "Contrastive prompting",
            "question": "What is contrastive prompting?",
            "reference_10_of_10": (
                "Contrastive prompting guides model behavior by presenting contrasting examples or reasoning paths—such "
                "as pairing desirable responses with undesirable ones, or comparing valid logic against common "
                "mistakes—so the model learns clear boundaries. Showing only positive examples can leave implicit "
                "pitfalls unclear, whereas contrasting good and bad cases explicitly highlights what to emulate and "
                "what to avoid. A strong explanation covers both the contrast itself and why it helps sharpen "
                "decision boundaries, formatting adherence, or factual accuracy. For example, a prompt might show a "
                "concise, grounded answer alongside a vague or speculative answer and explain why the first is "
                "preferred. This technique is useful for style alignment, error avoidance, safety guardrails, and "
                "nuanced classification tasks. A complete answer also notes that negative examples should be clearly "
                "labeled as undesirable so the model does not accidentally imitate the flawed pattern."
            ),
            "score": 10.0,
        },
        {
            "prompt_type": "Meta prompting",
            "question": "What is meta prompting?",
            "reference_10_of_10": (
                "Meta prompting treats prompt engineering or workflow design itself as the task, using a model to "
                "generate, critique, refine, or structure prompts, instructions, rubrics, or templates for itself or "
                "another system. Instead of directly solving only the end-user question, a meta prompt asks the model "
                "to analyze requirements, spot ambiguities, and produce a stronger prompt or reusable problem-solving "
                "framework. A thorough answer explains this higher-order focus on prompts and processes rather than "
                "just direct task answers. Meta prompting can help turn rough, underspecified ideas into structured "
                "instructions, design evaluation criteria, or adapt prompts for specialized downstream tasks. It can "
                "speed up prompt iteration and surface missing constraints early. A complete response also notes that "
                "generated prompts still benefit from human review and empirical testing, because a plausible-sounding "
                "meta-prompt can still introduce unnecessary complexity or miss domain edge cases."
            ),
            "score": 10.0,
        },
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "reference_10_of_10": (
                "Reverse prompting flips the usual question-and-answer flow by having the model ask targeted "
                "clarifying questions to gather missing context, constraints, or goals before generating a final "
                "solution. When a user request is broad, ambiguous, or missing key parameters, guessing immediately "
                "can produce an off-target answer. A strong explanation highlights this shift from immediate "
                "answering to structured requirement gathering through focused questions. For example, before "
                "recommending an architecture or drafting a plan, the model might ask about budget, scale, timeline, "
                "or target audience. Reverse prompting reduces wasted effort and misaligned outputs on open-ended or "
                "underspecified tasks. A complete answer also notes when to limit or skip it—such as when the user "
                "wants a quick best-effort answer immediately or when too many questions would slow down simple "
                "requests."
            ),
            "score": 10.0,
        },
    ]


def score_to_centered(score_0_10: float) -> float:
    """Map [0.0, 10.0] -> [-1.0, 1.0] so 5.0 sits at 0.0."""
    clamped = float(np.clip(score_0_10, 0.0, 10.0))
    return (clamped - 5.0) / 5.0


def centered_to_score(pred_centered: float) -> float:
    """Map [-1.0, 1.0] -> [0.0, 10.0]."""
    score = (float(pred_centered) * 5.0) + 5.0
    return float(np.clip(score, 0.0, 10.0))


def build_formatted_dataset() -> Dataset:
    rows = []
    for idx, item in enumerate(build_raw_samples()):
        text_a = f"Topic: {item['prompt_type']} | Question: {item['question']}"
        text_b = f"Reference (10/10): {item['reference_10_of_10']}"
        rows.append(
            {
                "id": idx,
                "prompt_type": item["prompt_type"],
                "text_a": text_a,
                "text_b": text_b,
                "label": score_to_centered(item["score"]),
            }
        )
    return Dataset.from_list(rows)


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds_10 = np.array([centered_to_score(p) for p in preds.squeeze(-1)])
    labels_10 = np.array([centered_to_score(y) for y in labels.squeeze()])
    mse = float(np.mean((labels_10 - preds_10) ** 2))
    mae = float(np.mean(np.abs(labels_10 - preds_10)))
    return {
        "mse": mse,
        "rmse": float(np.sqrt(mse)),
        "mae": mae,
    }


def get_precision_flags() -> dict:
    """Configure precision cleanly across CPU and CUDA GPUs."""
    if not torch.cuda.is_available():
        return {"use_cpu": True, "bf16": False, "fp16": False}
    bf16_supported = torch.cuda.is_bf16_supported()
    return {
        "use_cpu": False,
        "bf16": bf16_supported,
        "fp16": not bf16_supported,
    }


class CenteredRegressionTrainer(Trainer):
    """Uses SmoothL1Loss for regression training."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").float().view(-1)
        outputs = model(**inputs)
        logits = outputs.logits.float().view(-1)
        loss = nn.SmoothL1Loss(beta=0.2)(logits, labels)
        return (loss, outputs) if return_outputs else loss


def train_model():
    set_seed(SEED)
    precision = get_precision_flags()
    device_label = "GPU (CUDA)" if not precision["use_cpu"] else "CPU"
    print(f"Training on: {device_label} | precision={precision}")

    dataset = build_formatted_dataset()
    # 9 rows total (1 per prompt_type), so split without stratify_by_column
    split = dataset.train_test_split(test_size=2, seed=SEED)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize_fn(batch):
        return tokenizer(
            batch["text_a"],
            batch["text_b"],
            truncation=True,
            max_length=512,
            padding="max_length",
        )

    tokenized_train = split["train"].map(tokenize_fn, batched=True)
    tokenized_eval = split["test"].map(tokenize_fn, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=1,
        problem_type="regression",
    )

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=2,
        num_train_epochs=24,
        weight_decay=0.01,
        warmup_steps=0.1,
        max_grad_norm=1.0,
        load_best_model_at_end=True,
        metric_for_best_model="mae",
        greater_is_better=False,
        logging_steps=2,
        report_to="none",
        seed=SEED,
        use_cpu=precision["use_cpu"],
        bf16=precision["bf16"],
        fp16=precision["fp16"],
    )

    trainer = CenteredRegressionTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    eval_results = trainer.evaluate()
    print("\nValidation metrics (0-10 scale):", eval_results)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"Saved model and tokenizer to {OUTPUT_DIR}")


def score_responses(model_dir: str, test_cases: list[dict]):
    reference_lookup = {item["prompt_type"]: item for item in build_raw_samples()}

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    model.eval()

    print("\n--- Post-Training Evaluations Against 10/10 Reference (Scale: 0 - 10) ---")
    for i, case in enumerate(test_cases, start=1):
        ref_item = reference_lookup[case["prompt_type"]]
        text_a = f"Topic: {case['prompt_type']} | Question: {case['question']}"
        text_b = (
            f"Reference (10/10): {ref_item['reference_10_of_10']}\n"
            f"Candidate Response: {case['response']}"
        )
        inputs = tokenizer(
            text_a,
            text_b,
            return_tensors="pt",
            truncation=True,
            max_length=512,
        ).to(device)

        with torch.no_grad():
            pred_centered = model(**inputs).logits.squeeze().item()

        final_score = centered_to_score(pred_centered)
        print(f"\nSample {i} [{case['prompt_type']}]")
        print(f"  Question : {case['question']}")
        print(f"  Response : {case['response']}")
        print(f"  Score    : {final_score:.2f} / 10.00")


if __name__ == "__main__":
    train_model()

    sample_tests = [
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "response": "Instructs a model to perform a task directly using only task instructions and pre-trained knowledge, with zero in-context examples.",
        },
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "response": "Provides a single input-output example alongside instructions to clarify expected format, style, or mapping before answering.",
        },
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "response": "Samples multiple reasoning paths for a prompt and selects the most consistent final answer via majority consensus.",
        },
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "response": "Has the model ask targeted clarifying questions to gather missing requirements before generating a final solution.",
        },
    ]

    score_responses(OUTPUT_DIR, sample_tests)

In [ ]:
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

#MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
#MODEL_NAME = "BAAI/bge-small-en-v1.5"
MODEL_NAME ="sentence-transformers/all-MiniLM-L6-v2"


def build_reference_bank() -> dict[str, dict]:
    """
    9 comprehensive 10/10 reference answers (~150-200 words each), keyed by prompt_type.
    """
    samples = [
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "reference_10_of_10": (
                "Zero shot prompting is a technique in which a language model is asked to perform a task directly "
                "from natural language instructions without any labeled examples or demonstrations provided in the prompt. "
                "Instead of relying on in-context examples to show desired input-output patterns, the model draws on "
                "its pre-trained knowledge and instruction-following ability to interpret the task specification. "
                "A thorough answer explains both what is absent—zero examples—and what drives execution—clear task "
                "instructions, constraints, and existing model capabilities. It is especially useful when labeled "
                "examples are unavailable, expensive to create, or unnecessary for straightforward tasks such as "
                "basic classification, summarization, translation, or general question answering. However, zero shot "
                "prompting can struggle with complex formatting rules, niche domain conventions, or multi-step "
                "reasoning where demonstrations clarify expectations. Therefore, effective zero shot prompts state the "
                "objective unambiguously, specify output structure or tone when needed, and define boundaries so the "
                "model does not have to infer unstated requirements."
            ),
        },
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "reference_10_of_10": (
                "One shot prompting provides a single complete example—typically an input paired with its ideal "
                "output—alongside task instructions before asking the model to process a new input. That lone "
                "demonstration acts as a concrete blueprint showing desired formatting, style, level of detail, or "
                "transformation logic without requiring parameter updates or fine-tuning. A strong explanation highlights "
                "the distinction between zero examples and a single demonstration, as well as how one example reduces "
                "ambiguity when instructions alone might be interpreted in multiple ways. For instance, one example can "
                "illustrate a specific JSON schema, tone, or label convention immediately. At the same time, a complete "
                "answer notes its limitations: a single example may not cover diverse edge cases or complex variations, "
                "and a poorly chosen or unrepresentative example can bias the model toward a narrow pattern. Thus, "
                "one shot prompting works best when one clear, representative example captures the core task pattern."
            ),
        },
        {
            "prompt_type": "Few shot",
            "question": "What is few shot prompting?",
            "reference_10_of_10": (
                "Few shot prompting supplies multiple examples—usually several input-output pairs—inside the prompt "
                "context so the model can infer patterns, formatting rules, and reasoning style in-context without "
                "modifying model weights. A comprehensive answer emphasizes why multiple examples matter more than one: "
                "they help illustrate variety, edge cases, exceptions, and subtle distinctions across classes or "
                "formats. Few shot prompting leverages in-context learning while keeping the underlying model frozen, "
                "making it faster and more flexible than task-specific fine-tuning when requirements evolve. "
                "Good practice includes selecting diverse, accurate, and well-ordered examples that match the target "
                "task rather than noisy or contradictory ones. A complete explanation also recognizes trade-offs, "
                "such as increased token usage, higher latency and cost on long prompts, and potential sensitivity to "
                "example ordering or context-window limits. Overall, few shot prompting bridges direct instructions and "
                "fine-tuning when moderate guidance is needed."
            ),
        },
        {
            "prompt_type": "Adaptive",
            "question": "What is adaptive prompting?",
            "reference_10_of_10": (
                "Adaptive prompting dynamically adjusts prompt instructions, framing, detail level, or reasoning strategy "
                "based on changing context such as user expertise, task complexity, input characteristics, or feedback "
                "from earlier steps. Rather than applying a single static prompt to every request, an adaptive approach "
                "tailors how the model is guided so explanations or solutions fit the situation. A thorough answer "
                "explains what adapts—such as vocabulary depth, step-by-step scaffolding, format, or verification "
                "checks—and what triggers the adaptation, such as a beginner versus expert user, an ambiguous query, "
                "or an error detected during execution. It can be implemented through routing rules, dynamic prompt "
                "templates, or iterative refinement loops. Key benefits include better usability, clearer explanations, "
                "and more efficient use of tokens and reasoning effort across varied queries. A complete response also "
                "notes that added routing or adaptation logic introduces orchestration complexity that should be tested "
                "so behavior remains predictable."
            ),
        },
        {
            "prompt_type": "Prompt chaining",
            "question": "What is prompt chaining?",
            "reference_10_of_10": (
                "Prompt chaining decomposes a complex task into a sequence of smaller, focused prompts where the output "
                "of one step becomes the input—often structured—to the next step in a pipeline. Instead of forcing a "
                "single prompt to handle extraction, analysis, formatting, and validation all at once, chaining "
                "separates distinct sub-tasks into manageable stages. A strong explanation highlights sequential data "
                "flow between steps and why modularizing prompts improves reliability. Breaking work into stages makes "
                "it easier to inspect intermediate outputs, catch errors early, apply targeted instructions or "
                "constraints to each sub-task, and keep individual prompts concise and clear. It is widely used in "
                "multi-step workflows such as document summarization, research synthesis, data transformation, and code "
                "generation pipelines. A complete answer also notes trade-offs: chaining adds latency across "
                "sequential calls and can propagate errors if an earlier stage produces flawed output that is not "
                "checked before moving forward."
            ),
        },
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "reference_10_of_10": (
                "Self consistency prompting improves reliability on reasoning tasks by sampling multiple diverse "
                "reasoning paths for the same prompt and selecting the most consistent final answer, typically via "
                "majority vote. Rather than relying on a single greedy decoding path that might take a flawed "
                "intermediate step, this technique explores alternative ways to reach a solution and checks where "
                "independent paths converge. A thorough answer explains both parts of the mechanism: generating "
                "multiple reasoning traces (usually with non-zero sampling temperature) and aggregating their final "
                "answers by consensus. It is particularly effective for arithmetic, logic, and multi-step problem "
                "solving where multiple valid reasoning routes can lead to the same answer. A complete explanation "
                "also notes practical costs, since running multiple samples multiplies token usage and inference "
                "latency, and majority voting assumes the correct answer is more common across sampled paths than "
                "any single incorrect one."
            ),
        },
        {
            "prompt_type": "Contrastive prompting",
            "question": "What is contrastive prompting?",
            "reference_10_of_10": (
                "Contrastive prompting guides model behavior by presenting contrasting examples or reasoning paths—such "
                "as pairing desirable responses with undesirable ones, or comparing valid logic against common "
                "mistakes—so the model learns clear boundaries. Showing only positive examples can leave implicit "
                "pitfalls unclear, whereas contrasting good and bad cases explicitly highlights what to emulate and "
                "what to avoid. A strong explanation covers both the contrast itself and why it helps sharpen "
                "decision boundaries, formatting adherence, or factual accuracy. For example, a prompt might show a "
                "concise, grounded answer alongside a vague or speculative answer and explain why the first is "
                "preferred. This technique is useful for style alignment, error avoidance, safety guardrails, and "
                "nuanced classification tasks. A complete answer also notes that negative examples should be clearly "
                "labeled as undesirable so the model does not accidentally imitate the flawed pattern."
            ),
        },
        {
            "prompt_type": "Meta prompting",
            "question": "What is meta prompting?",
            "reference_10_of_10": (
                "Meta prompting treats prompt engineering or workflow design itself as the task, using a model to "
                "generate, critique, refine, or structure prompts, instructions, rubrics, or templates for itself or "
                "another system. Instead of directly solving only the end-user question, a meta prompt asks the model "
                "to analyze requirements, spot ambiguities, and produce a stronger prompt or reusable problem-solving "
                "framework. A thorough answer explains this higher-order focus on prompts and processes rather than "
                "just direct task answers. Meta prompting can help turn rough, underspecified ideas into structured "
                "instructions, design evaluation criteria, or adapt prompts for specialized downstream tasks. It can "
                "speed up prompt iteration and surface missing constraints early. A complete response also notes that "
                "generated prompts still benefit from human review and empirical testing, because a plausible-sounding "
                "meta-prompt can still introduce unnecessary complexity or miss domain edge cases."
            ),
        },
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "reference_10_of_10": (
                "Reverse prompting flips the usual question-and-answer flow by having the model ask targeted "
                "clarifying questions to gather missing context, constraints, or goals before generating a final "
                "solution. When a user request is broad, ambiguous, or missing key parameters, guessing immediately "
                "can produce an off-target answer. A strong explanation highlights this shift from immediate "
                "answering to structured requirement gathering through focused questions. For example, before "
                "recommending an architecture or drafting a plan, the model might ask about budget, scale, timeline, "
                "or target audience. Reverse prompting reduces wasted effort and misaligned outputs on open-ended or "
                "underspecified tasks. A complete answer also notes when to limit or skip it—such as when the user "
                "wants a quick best-effort answer immediately or when too many questions would slow down simple "
                "requests."
            ),
        },
    ]
    return {item["prompt_type"]: item for item in samples}


def logit_to_score_0_10(logit: float, low_logit: float = -6.0, high_logit: float = 4.0) -> float:
    """
    Map a cross-encoder logit onto a 0.0 - 10.0 scale.
    Tune low_logit and high_logit against your own sample responses if scores cluster too high or low.
    """
    scaled = (float(logit) - low_logit) / (high_logit - low_logit)
    return float(np.clip(scaled * 10.0, 0.0, 10.0))


def score_candidates_against_reference(test_cases: list[dict], model_name: str = MODEL_NAME):
    ref_bank = build_reference_bank()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    model.eval()

    print(f"Scoring on {device} using {model_name} (reference-only similarity)")
    for i, case in enumerate(test_cases, start=1):
        ref = ref_bank[case["prompt_type"]]
        question = case.get("question") or ref["question"]

        # Pair reference context with the candidate response
        text_a = f"{question} Reference answer: {ref['reference_10_of_10']}"
        text_b = f"Candidate answer: {case['response']}"

        inputs = tokenizer(
            text_a,
            text_b,
            return_tensors="pt",
            truncation=True,
            max_length=512,
        ).to(device)

        with torch.no_grad():
            raw_logit = model(**inputs).logits.squeeze().item()

        score_10 = logit_to_score_0_10(raw_logit)
        print(f"\nSample {i} [{case['prompt_type']}]")
        print(f"  Candidate : {case['response']}")
        print(f"  Raw Logit : {raw_logit:.4f}")
        print(f"  Score     : {score_10:.2f} / 10.00")


if __name__ == "__main__":
    sample_tests = [
        {
            "prompt_type": "Zero shot",
            "response": "Zero shot prompting tells the model to complete a task using only instructions with no examples given in the prompt.",
        },
        {
            "prompt_type": "One shot",
            "response": "One shot prompting includes one example pair to show the format before asking the model to answer a new input.",
        },
        {
            "prompt_type": "Self consistency prompting",
            "response": "It deletes the system prompt and randomly drops words from the user question.",
        },
        {
            "prompt_type": "Reverse prompting",
            "response": "Reverse prompting has the assistant ask clarifying questions to gather missing requirements before giving a final solution.",
        },
    ]

    score_candidates_against_reference(sample_tests)

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


def build_reference_bank() -> dict[str, dict]:
    """
    9 comprehensive 10/10 reference answers (~150-200 words each), keyed by prompt_type.
    """
    samples = [
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "reference_10_of_10": (
                "Zero shot prompting is a technique in which a language model is asked to perform a task directly "
                "from natural language instructions without any labeled examples or demonstrations provided in the prompt. "
                "Instead of relying on in-context examples to show desired input-output patterns, the model draws on "
                "its pre-trained knowledge and instruction-following ability to interpret the task specification. "
                "A thorough answer explains both what is absent—zero examples—and what drives execution—clear task "
                "instructions, constraints, and existing model capabilities. It is especially useful when labeled "
                "examples are unavailable, expensive to create, or unnecessary for straightforward tasks such as "
                "basic classification, summarization, translation, or general question answering. However, zero shot "
                "prompting can struggle with complex formatting rules, niche domain conventions, or multi-step "
                "reasoning where demonstrations clarify expectations. Therefore, effective zero shot prompts state the "
                "objective unambiguously, specify output structure or tone when needed, and define boundaries so the "
                "model does not have to infer unstated requirements."
            ),
        },
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "reference_10_of_10": (
                "One shot prompting provides a single complete example—typically an input paired with its ideal "
                "output—alongside task instructions before asking the model to process a new input. That lone "
                "demonstration acts as a concrete blueprint showing desired formatting, style, level of detail, or "
                "transformation logic without requiring parameter updates or fine-tuning. A strong explanation highlights "
                "the distinction between zero examples and a single demonstration, as well as how one example reduces "
                "ambiguity when instructions alone might be interpreted in multiple ways. For instance, one example can "
                "illustrate a specific JSON schema, tone, or label convention immediately. At the same time, a complete "
                "answer notes its limitations: a single example may not cover diverse edge cases or complex variations, "
                "and a poorly chosen or unrepresentative example can bias the model toward a narrow pattern. Thus, "
                "one shot prompting works best when one clear, representative example captures the core task pattern."
            ),
        },
        {
            "prompt_type": "Few shot",
            "question": "What is few shot prompting?",
            "reference_10_of_10": (
                "Few shot prompting supplies multiple examples—usually several input-output pairs—inside the prompt "
                "context so the model can infer patterns, formatting rules, and reasoning style in-context without "
                "modifying model weights. A comprehensive answer emphasizes why multiple examples matter more than one: "
                "they help illustrate variety, edge cases, exceptions, and subtle distinctions across classes or "
                "formats. Few shot prompting leverages in-context learning while keeping the underlying model frozen, "
                "making it faster and more flexible than task-specific fine-tuning when requirements evolve. "
                "Good practice includes selecting diverse, accurate, and well-ordered examples that match the target "
                "task rather than noisy or contradictory ones. A complete explanation also recognizes trade-offs, "
                "such as increased token usage, higher latency and cost on long prompts, and potential sensitivity to "
                "example ordering or context-window limits. Overall, few shot prompting bridges direct instructions and "
                "fine-tuning when moderate guidance is needed."
            ),
        },
        {
            "prompt_type": "Adaptive",
            "question": "What is adaptive prompting?",
            "reference_10_of_10": (
                "Adaptive prompting dynamically adjusts prompt instructions, framing, detail level, or reasoning strategy "
                "based on changing context such as user expertise, task complexity, input characteristics, or feedback "
                "from earlier steps. Rather than applying a single static prompt to every request, an adaptive approach "
                "tailors how the model is guided so explanations or solutions fit the situation. A thorough answer "
                "explains what adapts—such as vocabulary depth, step-by-step scaffolding, format, or verification "
                "checks—and what triggers the adaptation, such as a beginner versus expert user, an ambiguous query, "
                "or an error detected during execution. It can be implemented through routing rules, dynamic prompt "
                "templates, or iterative refinement loops. Key benefits include better usability, clearer explanations, "
                "and more efficient use of tokens and reasoning effort across varied queries. A complete response also "
                "notes that added routing or adaptation logic introduces orchestration complexity that should be tested "
                "so behavior remains predictable."
            ),
        },
        {
            "prompt_type": "Prompt chaining",
            "question": "What is prompt chaining?",
            "reference_10_of_10": (
                "Prompt chaining decomposes a complex task into a sequence of smaller, focused prompts where the output "
                "of one step becomes the input—often structured—to the next step in a pipeline. Instead of forcing a "
                "single prompt to handle extraction, analysis, formatting, and validation all at once, chaining "
                "separates distinct sub-tasks into manageable stages. A strong explanation highlights sequential data "
                "flow between steps and why modularizing prompts improves reliability. Breaking work into stages makes "
                "it easier to inspect intermediate outputs, catch errors early, apply targeted instructions or "
                "constraints to each sub-task, and keep individual prompts concise and clear. It is widely used in "
                "multi-step workflows such as document summarization, research synthesis, data transformation, and code "
                "generation pipelines. A complete answer also notes trade-offs: chaining adds latency across "
                "sequential calls and can propagate errors if an earlier stage produces flawed output that is not "
                "checked before moving forward."
            ),
        },
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "reference_10_of_10": (
                "Self consistency prompting improves reliability on reasoning tasks by sampling multiple diverse "
                "reasoning paths for the same prompt and selecting the most consistent final answer, typically via "
                "majority vote. Rather than relying on a single greedy decoding path that might take a flawed "
                "intermediate step, this technique explores alternative ways to reach a solution and checks where "
                "independent paths converge. A thorough answer explains both parts of the mechanism: generating "
                "multiple reasoning traces (usually with non-zero sampling temperature) and aggregating their final "
                "answers by consensus. It is particularly effective for arithmetic, logic, and multi-step problem "
                "solving where multiple valid reasoning routes can lead to the same answer. A complete explanation "
                "also notes practical costs, since running multiple samples multiplies token usage and inference "
                "latency, and majority voting assumes the correct answer is more common across sampled paths than "
                "any single incorrect one."
            ),
        },
        {
            "prompt_type": "Contrastive prompting",
            "question": "What is contrastive prompting?",
            "reference_10_of_10": (
                "Contrastive prompting guides model behavior by presenting contrasting examples or reasoning paths—such "
                "as pairing desirable responses with undesirable ones, or comparing valid logic against common "
                "mistakes—so the model learns clear boundaries. Showing only positive examples can leave implicit "
                "pitfalls unclear, whereas contrasting good and bad cases explicitly highlights what to emulate and "
                "what to avoid. A strong explanation covers both the contrast itself and why it helps sharpen "
                "decision boundaries, formatting adherence, or factual accuracy. For example, a prompt might show a "
                "concise, grounded answer alongside a vague or speculative answer and explain why the first is "
                "preferred. This technique is useful for style alignment, error avoidance, safety guardrails, and "
                "nuanced classification tasks. A complete answer also notes that negative examples should be clearly "
                "labeled as undesirable so the model does not accidentally imitate the flawed pattern."
            ),
        },
        {
            "prompt_type": "Meta prompting",
            "question": "What is meta prompting?",
            "reference_10_of_10": (
                "Meta prompting treats prompt engineering or workflow design itself as the task, using a model to "
                "generate, critique, refine, or structure prompts, instructions, rubrics, or templates for itself or "
                "another system. Instead of directly solving only the end-user question, a meta prompt asks the model "
                "to analyze requirements, spot ambiguities, and produce a stronger prompt or reusable problem-solving "
                "framework. A thorough answer explains this higher-order focus on prompts and processes rather than "
                "just direct task answers. Meta prompting can help turn rough, underspecified ideas into structured "
                "instructions, design evaluation criteria, or adapt prompts for specialized downstream tasks. It can "
                "speed up prompt iteration and surface missing constraints early. A complete response also notes that "
                "generated prompts still benefit from human review and empirical testing, because a plausible-sounding "
                "meta-prompt can still introduce unnecessary complexity or miss domain edge cases."
            ),
        },
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "reference_10_of_10": (
                "Reverse prompting flips the usual question-and-answer flow by having the model ask targeted "
                "clarifying questions to gather missing context, constraints, or goals before generating a final "
                "solution. When a user request is broad, ambiguous, or missing key parameters, guessing immediately "
                "can produce an off-target answer. A strong explanation highlights this shift from immediate "
                "answering to structured requirement gathering through focused questions. For example, before "
                "recommending an architecture or drafting a plan, the model might ask about budget, scale, timeline, "
                "or target audience. Reverse prompting reduces wasted effort and misaligned outputs on open-ended or "
                "underspecified tasks. A complete answer also notes when to limit or skip it—such as when the user "
                "wants a quick best-effort answer immediately or when too many questions would slow down simple "
                "requests."
            ),
        },
    ]
    return {item["prompt_type"]: item for item in samples}


def embed_texts(
    texts: list[str],
    tokenizer: AutoTokenizer,
    model: AutoModel,
    device: torch.device,
    max_length: int = 256,
) -> torch.Tensor:
    """Compute L2-normalized mean-pooled embeddings for a list of strings."""
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model(**encoded)
        token_embeddings = outputs.last_hidden_state
        mask = encoded["attention_mask"].unsqueeze(-1).expand(token_embeddings.size()).float()
        summed = torch.sum(token_embeddings * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        pooled = summed / counts
        return F.normalize(pooled, p=2, dim=1)


def similarity_to_score_0_10(sim: float, low_sim: float = 0.15, high_sim: float = 0.85) -> float:
    """Map cosine similarity onto a 0.0 - 10.0 scale."""
    scaled = (float(sim) - low_sim) / (high_sim - low_sim)
    return float(np.clip(scaled * 10.0, 0.0, 10.0))


def score_candidates_against_reference(test_cases: list[dict], model_name: str = MODEL_NAME):
    ref_bank = build_reference_bank()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    # Precompute reference embeddings once
    prompt_types = list(ref_bank.keys())
    ref_texts = [ref_bank[pt]["reference_10_of_10"] for pt in prompt_types]
    ref_embs = embed_texts(ref_texts, tokenizer, model, device)
    ref_emb_map = {pt: ref_embs[idx] for idx, pt in enumerate(prompt_types)}

    cand_texts = [case["response"] for case in test_cases]
    cand_embs = embed_texts(cand_texts, tokenizer, model, device)

    print(f"Scoring on {device} using {model_name} (cosine similarity)")
    for i, case in enumerate(test_cases):
        pt = case["prompt_type"]
        # Cosine similarity of two L2-normalized vectors is their dot product (scalar float)
        cos_sim = torch.dot(ref_emb_map[pt], cand_embs[i]).item()
        score_10 = similarity_to_score_0_10(cos_sim)

        print(f"\nSample {i + 1} [{pt}]")
        print(f"  Candidate  : {case['response']}")
        print(f"  Cosine Sim : {cos_sim:.4f}")
        print(f"  Score      : {score_10:.2f} / 10.00")


if __name__ == "__main__":
    sample_tests = [
        {
            "prompt_type": "Zero shot",
            "response": "Zero shot prompting tells the model to complete a task using only instructions with no examples given in the prompt.",
        },
        {
            "prompt_type": "One shot",
            "response": "One shot prompting includes one example pair to show the format before asking the model to answer a new input.",
        },
        {
            "prompt_type": "Self consistency prompting",
            "response": "It deletes the system prompt and randomly drops words from the user question.",
        },
        {
            "prompt_type": "Reverse prompting",
            "response": "Reverse prompting has the assistant ask clarifying questions to gather missing requirements before giving a final solution.",
        },
    ]

    score_candidates_against_reference(sample_tests)